# 1 ) Feature Engineering

## 1.1 Feature Engineering Objective

The objective of feature engineering is to transform raw borrower and credit history data into meaningful predictive features that improve machine learning model performance for loan default and recovery prediction.

This phase focuses on creating financial ratios, debt indicators, repayment behavior features, behavioral risk variables, and borrower-level aggregated features using application and bureau datasets.

In [ ]:
# IMPORT LIBRARIES
# Data handling and numerical operations
import pandas as pd
import numpy as np
# Ignore unnecessary warnings
import warnings
warnings.filterwarnings('ignore')

# 2 ) Load Cleaned Datasets

## 2.1 Load Application Dataset

In [ ]:
# LOAD CLEANED APPLICATION DATASET
application_train = pd.read_csv('application_train_cleaned.csv')
# Dataset shape
print(f'Application Dataset Shape : {application_train.shape}')

Application Dataset Shape : (307511, 98)


## 2.2 Load Bureau Dataset

In [ ]:
# LOAD CLEANED BUREAU DATASET
bureau = pd.read_csv('bureau_cleaned.csv')
# Dataset shape
print(f'Bureau Dataset Shape : {bureau.shape}')

Bureau Dataset Shape : (1716428, 17)


# 3 ) Low-Information Feature Removal

## 3.1 Check Document Features

In [ ]:
# CHECK DOCUMENT FLAG FEATURES

# Select document flag columns

doc_flags = [
    col for col in application_train.columns
        if 'FLAG_DOCUMENT' in col
]
# Check value distribution
for col in doc_flags:
    print('=' * 50)
    print(f'Column : {col}')
    print(f'Unique Values : {application_train[col].nunique()}')
    print('\nValue Counts :')
    print(application_train[col].value_counts())
    print('\n')

Column : FLAG_DOCUMENT_2
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_2
0    307498
1        13
Name: count, dtype: int64


Column : FLAG_DOCUMENT_3
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_3
1    218340
0     89171
Name: count, dtype: int64


Column : FLAG_DOCUMENT_4
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_4
0    307486
1        25
Name: count, dtype: int64


Column : FLAG_DOCUMENT_5
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_5
0    302863
1      4648
Name: count, dtype: int64


Column : FLAG_DOCUMENT_6
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_6
0    280433
1     27078
Name: count, dtype: int64


Column : FLAG_DOCUMENT_7
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_7
0    307452
1        59
Name: count, dtype: int64


Column : FLAG_DOCUMENT_8
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_8
0    282487
1     25024
Name: count, dtype: int64


Column : FLAG_DOCUMENT_9
Unique Values : 2

Value Counts :
FLAG_DOCUMENT_9
0    306313
1      1198
Name: count,


-  Most FLAG_DOCUMENT features are highly sparse and imbalanced.
 - Majority of applicants have value 0 for most document flags.
- FLAG_DOCUMENT_3 has the strongest presence among all document features.
 - Several document flags contain very few positive cases and may contribute - - limited predictive power individually.
-  Aggregating document flags into TOTAL_DOCUMENTS_SUBMITTED can provide a more meaningful behavioral feature.

## 3.2 Remove Document Features

In [ ]:
# REMOVE LOW-INFORMATION DOCUMENT FEATURES
# Dataset shape before removal
print(f'Before Shape : {application_train.shape}')
# Low-variance document features
remove_docs = ['FLAG_DOCUMENT_2','FLAG_DOCUMENT_4','FLAG_DOCUMENT_7','FLAG_DOCUMENT_10','FLAG_DOCUMENT_12','FLAG_DOCUMENT_17']
# Remove columns
application_train.drop(columns=remove_docs,inplace=True,errors='ignore')
# Dataset shape after removal
print(f'After Shape : {application_train.shape}')
print('\nLow-information document features removed successfully!')

Before Shape : (307511, 98)
After Shape : (307511, 92)

Low-information document features removed successfully!


- Removed extremely sparse and low-information document features with near-zero variance.
- These features contained very few positive records and were unlikely to contribute meaningful predictive power.
- Feature reduction decreased dimensionality from 98 to 92 columns.
- Removing low-information features helps reduce noise and improve model efficiency.

## 3.3 Check Mobile Features

In [ ]:
# CHECK MOBILE FLAG FEATURES
# Mobile-related flag columns
mobile_flags = ['FLAG_MOBIL','FLAG_CONT_MOBILE']
# Check value distribution
for col in mobile_flags:
    print('=' * 50)
    print(f'Column : {col}')
    print(f'Unique Values : {application_train[col].nunique()}')
    print('\nValue Counts :')
    print(application_train[col].value_counts())
    print('\n')

Column : FLAG_MOBIL
Unique Values : 2

Value Counts :
FLAG_MOBIL
1    307510
0         1
Name: count, dtype: int64


Column : FLAG_CONT_MOBILE
Unique Values : 2

Value Counts :
FLAG_CONT_MOBILE
1    306937
0       574
Name: count, dtype: int64




- FLAG_MOBIL is nearly constant, with only one record having value 0, indicating extremely low variance.
- FLAG_CONT_MOBILE is also highly imbalanced, with the majority of applicants having value 1.
- FLAG_MOBIL is unlikely to provide meaningful predictive power and may be considered for removal.
- FLAG_CONT_MOBILE may still retain limited behavioral information despite imbalance.

## 3.4 Remove Mobile Features

In [ ]:
# REMOVE LOW-VARIANCE MOBILE FEATURE
# Dataset shape before removal
print(f'Before Shape : {application_train.shape}')
# Remove FLAG_MOBIL
application_train.drop(columns=['FLAG_MOBIL'],inplace=True,errors='ignore')
# Dataset shape after removal
print(f'After Shape : {application_train.shape}')
print('\nFLAG_MOBIL removed successfully!')

Before Shape : (307511, 92)
After Shape : (307511, 91)

FLAG_MOBIL removed successfully!


- FLAG_MOBIL showed near-zero variance, with almost all applicants having value 1.
- The feature contained minimal discriminatory information and was unlikely to improve model performance.
- Removing FLAG_MOBIL reduced unnecessary dimensionality and helped simplify the feature space.
- Dataset dimensionality decreased from 92 to 91 columns after removal.

# 4 ) Financial Ratio Features

## 4.1 Credit & Income Features

In [ ]:
# FINANCIAL RATIO FEATURES
# CREDIT TO INCOME RATIO
# Measures loan burden
# relative to applicant income
application_train['CREDIT_INCOME_RATIO'] = (application_train['AMT_CREDIT']/(application_train['AMT_INCOME_TOTAL']+1))
# ANNUITY TO INCOME RATIO
# Measures EMI burden
# relative to applicant income
application_train['ANNUITY_INCOME_RATIO']=(application_train['AMT_ANNUITY']/(application_train['AMT_INCOME_TOTAL']+1))
# GOODS TO CREDIT RATIO
# Measures proportion of goods value
# covered by credit
application_train['GOODS_CREDIT_RATIO'](application_train['AMT_GOODS_PRICE']/(application_train['AMT_CREDIT']+1))
# DISPLAY CREATED FEATURES
application_train[['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','GOODS_CREDIT_RATIO']].head()

,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,GOODS_CREDIT_RATIO
0,2.007879,0.121977,0.863259
1,4.790732,0.132216,0.873210
2,1.999970,0.099999,0.999993
3,2.316150,0.219898,0.949842
4,4.222187,0.179961,0.999998


- CREDIT_INCOME_RATIO:
Measures loan burden relative to applicant income.
Higher values may indicate higher repayment risk.

- ANNUITY_INCOME_RATIO:
Measures EMI burden relative to applicant income.
Higher EMI obligations may reduce repayment capacity.

- GOODS_CREDIT_RATIO:Measures proportion of goods value covered by credit.
Helps identify financing structure and over-financing risk.

## 4.2 Family Financial Features

In [ ]:
# INCOME PER FAMILY MEMBER
# Measures available income per family member
application_train['INCOME_PER_PERSON']=(application_train['AMT_INCOME_TOTAL']/(application_train['CNT_FAM_MEMBERS']+1))
# CHILDREN TO FAMILY RATIO
# Measures dependency burden in the family
application_train['CHILDREN_RATIO']=(application_train['CNT_CHILDREN']/(application_train['CNT_FAM_MEMBERS']+1))
# Display created features
application_train[['INCOME_PER_PERSON','CHILDREN_RATIO']].head()

,INCOME_PER_PERSON,CHILDREN_RATIO
0,101250.0,0.0
1,90000.0,0.0
2,33750.0,0.0
3,45000.0,0.0
4,60750.0,0.0


- INCOME_PER_PERSON:
 Captures household-level income distribution
 by measuring available income per family member.
 Lower values may indicate higher financial stress
 and reduced repayment flexibility.

- CHILDREN_RATIO
 Captures dependency burden within the household
 by measuring the proportion of children
 relative to total family members.
 Higher dependency may negatively impact
 repayment capacity and financial stability.

## 4.3 External Credit Score Features

In [ ]:
# EXT SOURCE COMBINED FEATURE
application_train['EXT_SOURCE_MEAN'] =application_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)
# Display feature
application_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','EXT_SOURCE_MEAN']].head()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,EXT_SOURCE_MEAN
0,0.083037,0.262949,0.139376,0.161787
1,0.311267,0.622246,NaN,0.466757
2,NaN,0.555912,0.729567,0.642739
3,NaN,0.650442,NaN,0.650442
4,NaN,0.322738,NaN,0.322738


- Created EXT_SOURCE_MEAN by averaging EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3 scores.
- The combined feature captures overall external creditworthiness more effectively than individual scores.
- Mean aggregation helps reduce the impact of missing values across external score features.
- EXT_SOURCE_MEAN is expected to provide strong predictive power for default risk modeling.

## 4.4 Collateral Features

In [ ]:
# HAS COLLATERAL FEATURE
# Indicates whether borrower owns
# car or real estate property
application_train['HAS_COLLATERAL']=((application_train['FLAG_OWN_CAR'] == 'Y')|(application_train['FLAG_OWN_REALTY'] == 'Y')).astype(int)
application_train[['FLAG_OWN_CAR','FLAG_OWN_REALTY','HAS_COLLATERAL']].head()

,FLAG_OWN_CAR,FLAG_OWN_REALTY,HAS_COLLATERAL
0,N,Y,1
1,N,N,0
2,Y,Y,1
3,N,Y,1
4,N,Y,1


- Created HAS_COLLATERAL feature to identify borrowers owning either a car or real estate property.
- The feature captures asset ownership, which may indicate better financial stability and repayment capacity.
- Combining multiple ownership indicators into a single feature helps reduce feature redundancy.
- HAS_COLLATERAL can serve as a useful behavioral and financial risk indicator in credit risk modeling.

## 4.5 External Credit Score Variability

In [ ]:
# EXT SOURCE STANDARD DEVIATION
# Measures variation among
# external credit scores
application_train['EXT_SOURCE_STD'] = (application_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].std(axis=1))
application_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','EXT_SOURCE_STD']].head()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,EXT_SOURCE_STD
0,0.083037,0.262949,0.139376,0.092026
1,0.311267,0.622246,NaN,0.219895
2,NaN,0.555912,0.729567,0.122792
3,NaN,0.650442,NaN,NaN
4,NaN,0.322738,NaN,NaN


- Created EXT_SOURCE_STD to measure variation across external credit score sources.
- Lower standard deviation indicates more consistent external credit behavior, while higher values may indicate uncertainty or risk.
- The feature captures score stability, which may provide additional predictive signal beyond average credit scores.
- Missing values occur when insufficient external score information is available for a borrower.

## 4.6 Missing Credit Score Indicator

In [ ]:
# EXT SOURCE 1 MISSING FLAG
# Indicates whether
# EXT_SOURCE_1 is missing
application_train['EXT_SOURCE_1_MISSING'] = (application_train['EXT_SOURCE_1'].isnull().astype(int))
application_train[['EXT_SOURCE_1','EXT_SOURCE_1_MISSING']].head()

,EXT_SOURCE_1,EXT_SOURCE_1_MISSING
0,0.083037,0
1,0.311267,0
2,NaN,1
3,NaN,1
4,NaN,1


- Created EXT_SOURCE_1_MISSING flag to capture missingness in EXT_SOURCE_1.
- Missing external credit scores may themselves contain important behavioral or risk-related information.
- Converting missingness into a binary feature allows the model to learn patterns associated with unavailable credit data.
- This feature is especially useful because EXT_SOURCE_1 contains a high percentage of missing values.

## 4.7 Financial Burden Features

In [ ]:
# ============================================================
# CREDIT TO ANNUITY RATIO
# ============================================================
# Measures loan amount
# relative to EMI burden
application_train['CREDIT_ANNUITY_RATIO'] = (application_train['AMT_CREDIT'] / (application_train['AMT_ANNUITY'] + 1))
application_train[['AMT_CREDIT','AMT_ANNUITY','CREDIT_ANNUITY_RATIO']].head()

,AMT_CREDIT,AMT_ANNUITY,CREDIT_ANNUITY_RATIO
0,406597.5,24700.5,16.460438
1,1293502.5,35698.5,36.233070
2,135000.0,6750.0,19.997037
3,312682.5,29686.5,10.532463
4,513000.0,21865.5,23.460545


## 4.8 Credit Inquiry Features

In [ ]:
# ============================================================
# TOTAL CREDIT BUREAU INQUIRIES
# ============================================================
# Measures total number
# of bureau credit inquiries
application_train['TOTAL_INQUIRIES'] = (application_train['AMT_REQ_CREDIT_BUREAU_HOUR'] + application_train['AMT_REQ_CREDIT_BUREAU_DAY'] + application_train['AMT_REQ_CREDIT_BUREAU_WEEK'] + application_train['AMT_REQ_CREDIT_BUREAU_MON'] + application_train['AMT_REQ_CREDIT_BUREAU_QRT'] + application_train['AMT_REQ_CREDIT_BUREAU_YEAR'])
application_train[['TOTAL_INQUIRIES']].head()

,TOTAL_INQUIRIES
0,1.0
1,0.0
2,0.0
3,1.0
4,0.0


In [ ]:
# ============================================================
# RECENT INQUIRY RATIO
# ============================================================
# Measures recent inquiry intensity
application_train['RECENT_INQUIRY_RATIO'] = ((application_train['AMT_REQ_CREDIT_BUREAU_DAY'] + application_train['AMT_REQ_CREDIT_BUREAU_WEEK'] + application_train['AMT_REQ_CREDIT_BUREAU_MON']) / (application_train['TOTAL_INQUIRIES'] + 1))
application_train[['TOTAL_INQUIRIES','RECENT_INQUIRY_RATIO']].head()

,TOTAL_INQUIRIES,RECENT_INQUIRY_RATIO
0,1.0,0.0
1,0.0,0.0
2,0.0,0.0
3,1.0,0.0
4,0.0,0.0


# 5 ) Document-Based Features

In [ ]:
# ============================================================
# BUREAU MERGE VALIDATION
# ============================================================
bureau_features = ['ACTIVE_LOAN_RATIO','CLOSED_LOAN_RATIO','OVERDUE_PER_LOAN','PROLONGED_LOAN_RATIO','CREDIT_UTILIZATION_RATIO']
print(application_train[bureau_features].head())
print('\n')
print('Bureau features successfully merged!')

   ACTIVE_LOAN_RATIO  CLOSED_LOAN_RATIO  OVERDUE_PER_LOAN  \
0           0.375000           0.500000               0.0   
1           0.222222           0.666667               0.0   
2           0.200000           0.600000               0.0   
3           0.000000           0.666667               0.0   
4           0.500000           0.250000               0.0   

   PROLONGED_LOAN_RATIO  CREDIT_UTILIZATION_RATIO  
0                   0.0                  0.410555  
1                   0.0                  0.284121  
2                   0.0                  0.000000  
3                   0.0                  0.000000  
4                   0.0                  0.864990  


Bureau features successfully merged!


## 5.1 Total Documents Submitted

In [ ]:
# TOTAL DOCUMENTS SUBMITTED
# Select remaining document flag columns
doc_flags = [col for col in application_train.columns if 'FLAG_DOCUMENT' in col]
# Count total submitted documents
application_train['TOTAL_DOCUMENTS_SUBMITTED'] = application_train[doc_flags].sum(axis=1)
# Display feature
application_train[['TOTAL_DOCUMENTS_SUBMITTED']].head()

,TOTAL_DOCUMENTS_SUBMITTED
0,1
1,1
2,0
3,1
4,1


- Created TOTAL_DOCUMENTS_SUBMITTED by aggregating all remaining document flag features.
- The feature captures overall document submission behavior of borrowers.
- Higher document submission counts may indicate better verification completeness and lower application risk.
- Aggregating sparse document features into a single numerical feature helps reduce dimensionality and improve feature usefulness.

# 6 ) Bureau-Based Features

## 6.1 Bureau Risk Ratios

In [ ]:
# BUREAU DEBT RATIO
# Measures remaining debt
# relative to total borrowed credit
application_train['BUREAU_DEBT_RATIO'] = (application_train['AMT_CREDIT_SUM_DEBT_sum'] / (application_train['AMT_CREDIT_SUM_sum'] + 1))
# AVERAGE CREDIT PER LOAN
# Measures average credit amount
# per previous loan
application_train['AVG_CREDIT_PER_LOAN'] = (application_train['AMT_CREDIT_SUM_sum'] / (application_train['SK_ID_BUREAU_count'] + 1))
# DEBT PER LOAN
# Measures average remaining debt
# per previous loan
application_train['DEBT_PER_LOAN'] = (application_train['AMT_CREDIT_SUM_DEBT_sum'] / (application_train['SK_ID_BUREAU_count'] + 1))
# Display engineered bureau features
application_train[['BUREAU_DEBT_RATIO','AVG_CREDIT_PER_LOAN','DEBT_PER_LOAN']].head()

,BUREAU_DEBT_RATIO,AVG_CREDIT_PER_LOAN,DEBT_PER_LOAN
0,0.284121,96117.284444,27309.0
1,0.000000,203480.100000,0.0
2,0.000000,63012.600000,0.0
3,0.000000,0.000000,0.0
4,0.000000,73125.000000,0.0


- Created bureau-based behavioral risk features using aggregated historical credit bureau information.
- BUREAU_DEBT_RATIO captures the proportion of remaining debt relative to total borrowed credit.
- AVG_CREDIT_PER_LOAN measures the average historical credit exposure per previous loan.
- DEBT_PER_LOAN estimates the average outstanding debt burden across previous loans.
- These features help represent borrower credit utilization, repayment burden, and historical borrowing behavior.

## 6.2 Advanced Bureau Behavioral Features

In [ ]:
# ADVANCED BUREAU AGGREGATION
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
# Total loans
bureau_total_loans = ('SK_ID_BUREAU','count'),
# Active loans
bureau_active_loans = ('CREDIT_ACTIVE',lambda x: (x == 'Active').sum()),
# Closed loans
bureau_closed_loans = ('CREDIT_ACTIVE',lambda x: (x == 'Closed').sum()),
# Total credit
bureau_total_credit = ('AMT_CREDIT_SUM','sum'),
# Total debt
bureau_total_debt = ('AMT_CREDIT_SUM_DEBT','sum'),
# Total overdue amount
bureau_total_overdue = ('AMT_CREDIT_SUM_OVERDUE','sum'),
# Loan prolong count
bureau_total_prolonged = ('CNT_CREDIT_PROLONG','sum')
).reset_index()
# Display aggregated bureau dataset
bureau_agg.head()

,SK_ID_CURR,bureau_total_loans,bureau_active_loans,bureau_closed_loans,bureau_total_credit,bureau_total_debt,bureau_total_overdue,bureau_total_prolonged
0,100001,7,3,4,1453365.000,596686.5,0.0,0
1,100002,8,2,6,865055.564,245781.0,0.0,0
2,100003,4,1,3,1017400.500,0.0,0.0,0
3,100004,2,0,2,189037.800,0.0,0.0,0
4,100005,3,2,1,657126.000,568408.5,0.0,0


- Performed advanced bureau-level aggregation using historical credit bureau records grouped by SK_ID_CURR.
- Created behavioral credit features related to total loans, active loans, closed loans, debt exposure, overdue amounts, and prolonged credits.
- Aggregated features help summarize borrower credit history and long-term repayment behavior.
- Bureau-based aggregations are expected to provide strong predictive signals for credit risk and recovery probability modeling.

In [ ]:
# ============================================================
# ADVANCED BUREAU BEHAVIORAL FEATURES
# ============================================================
# ACTIVE LOAN RATIO
# Measures proportion of active loans
# relative to total historical loans
application_train['ACTIVE_LOAN_RATIO'] = (bureau_agg['bureau_active_loans'] / (bureau_agg['bureau_total_loans'] + 1))
# ============================================================
# CLOSED LOAN RATIO
# ============================================================
# Measures proportion of closed loans
# relative to total historical loans
application_train['CLOSED_LOAN_RATIO'] = (bureau_agg['bureau_closed_loans'] / (bureau_agg['bureau_total_loans'] + 1))
# ============================================================
# OVERDUE PER LOAN
# ============================================================
# Measures average overdue amount
# per previous loan
application_train['OVERDUE_PER_LOAN'] = (bureau_agg['bureau_total_overdue'] / (bureau_agg['bureau_total_loans'] + 1))
# ============================================================
# PROLONGED LOAN RATIO
# ============================================================
# Measures frequency of loan prolongation
application_train['PROLONGED_LOAN_RATIO'] = (bureau_agg['bureau_total_prolonged'] / (bureau_agg['bureau_total_loans'] + 1))
# ============================================================
# CREDIT UTILIZATION RATIO
# ============================================================
# Measures utilized debt
# relative to total historical credit
application_train['CREDIT_UTILIZATION_RATIO'] = (bureau_agg['bureau_total_debt'] / (bureau_agg['bureau_total_credit'] + 1))
# ============================================================
# DISPLAY FEATURES
# ============================================================
application_train[['ACTIVE_LOAN_RATIO','CLOSED_LOAN_RATIO','OVERDUE_PER_LOAN','PROLONGED_LOAN_RATIO','CREDIT_UTILIZATION_RATIO']].head()

,ACTIVE_LOAN_RATIO,CLOSED_LOAN_RATIO,OVERDUE_PER_LOAN,PROLONGED_LOAN_RATIO,CREDIT_UTILIZATION_RATIO
0,0.375000,0.500000,0.0,0.0,0.410555
1,0.222222,0.666667,0.0,0.0,0.284121
2,0.200000,0.600000,0.0,0.0,0.000000
3,0.000000,0.666667,0.0,0.0,0.000000
4,0.500000,0.250000,0.0,0.0,0.864990


- Created advanced bureau behavioral features to capture borrower repayment behavior and historical credit activity.
- ACTIVE_LOAN_RATIO and CLOSED_LOAN_RATIO represent the proportion of active and closed loans relative to total historical loans.
- OVERDUE_PER_LOAN measures average overdue burden per previous loan account.
- PROLONGED_LOAN_RATIO captures the frequency of loan prolongation behavior, which may indicate repayment difficulty.
- CREDIT_UTILIZATION_RATIO estimates the proportion of utilized debt relative to total historical credit exposure.
- These behavioral features provide valuable insights into borrower credit management and financial risk patterns.

# 7 ) Behavioral Features

## 7.1 Employment Features

In [ ]:
# ============================================================
# AGE IN YEARS
# ============================================================
# Convert negative days into positive years
application_train['AGE_YEARS'] = (abs(application_train['DAYS_BIRTH']) / 365)
# ============================================================
# EMPLOYMENT IN YEARS
# ============================================================
# Convert employment duration into years
application_train['EMPLOYMENT_YEARS'] = (abs(application_train['DAYS_EMPLOYED']) / 365)
# ============================================================
# EMPLOYMENT TO AGE RATIO
# ============================================================
# Measures work experience
# relative to applicant age
application_train['EMPLOYMENT_AGE_RATIO'] = (application_train['EMPLOYMENT_YEARS'] / (application_train['AGE_YEARS'] + 1))
# Display engineered features
application_train[['AGE_YEARS','EMPLOYMENT_YEARS','EMPLOYMENT_AGE_RATIO']].head()

,AGE_YEARS,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO
0,25.920548,1.745205,0.064828
1,45.931507,3.254795,0.069352
2,52.180822,0.616438,0.011591
3,52.068493,8.326027,0.156892
4,54.608219,8.323288,0.149677


- Created AGE_YEARS and EMPLOYMENT_YEARS by converting negative day-based features into positive yearly values.
- EMPLOYMENT_AGE_RATIO captures the proportion of work experience relative to applicant age.
- These features help represent applicant maturity, career stability, and long-term employment behavior.
- Higher employment duration and stable employment-age relationships may indicate lower financial risk and improved repayment capability.

In [ ]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AVG_CREDIT_PER_LOAN,DEBT_PER_LOAN,ACTIVE_LOAN_RATIO,CLOSED_LOAN_RATIO,OVERDUE_PER_LOAN,PROLONGED_LOAN_RATIO,CREDIT_UTILIZATION_RATIO,AGE_YEARS,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,96117.284444,27309.0,0.375000,0.500000,0.0,0.0,0.410555,25.920548,1.745205,0.064828
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,203480.100000,0.0,0.222222,0.666667,0.0,0.0,0.284121,45.931507,3.254795,0.069352
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,63012.600000,0.0,0.200000,0.600000,0.0,0.0,0.000000,52.180822,0.616438,0.011591
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.000000,0.0,0.000000,0.666667,0.0,0.0,0.000000,52.068493,8.326027,0.156892
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,73125.000000,0.0,0.500000,0.250000,0.0,0.0,0.864990,54.608219,8.323288,0.149677


## 7.2 Social Risk Features

In [ ]:
# TOTAL SOCIAL DEFAULTS
# Combine social default indicators
application_train['TOTAL_SOCIAL_DEFAULTS'] = (application_train['DEF_30_CNT_SOCIAL_CIRCLE'] + application_train['DEF_60_CNT_SOCIAL_CIRCLE'])
# ============================================================
# TOTAL SOCIAL OBSERVATIONS
# ============================================================
# Combine observed social risk indicators
application_train['TOTAL_SOCIAL_OBS'] = (application_train['OBS_30_CNT_SOCIAL_CIRCLE'] + application_train['OBS_60_CNT_SOCIAL_CIRCLE'])
# Display engineered features
application_train[['TOTAL_SOCIAL_DEFAULTS','TOTAL_SOCIAL_OBS']].head()

,TOTAL_SOCIAL_DEFAULTS,TOTAL_SOCIAL_OBS
0,4.0,4.0
1,0.0,2.0
2,0.0,0.0
3,0.0,4.0
4,0.0,0.0


- Created TOTAL_SOCIAL_DEFAULTS by combining social default indicators from 30-day and 60-day social circles.
- Created TOTAL_SOCIAL_OBS to capture the total number of observed social risk indicators surrounding the applicant.
- These features help represent social-risk exposure and potential behavioral influence from the applicant’s social environment.
- Higher social default and observation counts may indicate increased financial risk and repayment uncertainty.

# 8 ) Missing Value Handling

## 8.1  Check Missing Values

In [ ]:
# ============================================================
# CHECK MISSING VALUES
# ============================================================
missing_values = application_train.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)

COMMONAREA_AVG                 214865
NONLIVINGAPARTMENTS_AVG        213514
LIVINGAPARTMENTS_AVG           210199
FLOORSMIN_AVG                  208642
YEARS_BUILD_AVG                204488
LANDAREA_AVG                   182590
BASEMENTAREA_AVG               179943
EXT_SOURCE_1                   173378
NONLIVINGAREA_AVG              169682
ELEVATORS_AVG                  163891
WALLSMATERIAL_MODE             156341
APARTMENTS_AVG                 156061
ENTRANCES_AVG                  154828
LIVINGAREA_AVG                 154350
FLOORSMAX_AVG                  153020
YEARS_BEGINEXPLUATATION_AVG    150007
TOTALAREA_MODE                 148431
OCCUPATION_TYPE                 96391
EXT_SOURCE_3                    60965
EMPLOYMENT_AGE_RATIO            55374
DAYS_EMPLOYED                   55374
EMPLOYMENT_YEARS                55374
AMT_REQ_CREDIT_BUREAU_QRT       41519
AMT_REQ_CREDIT_BUREAU_MON       41519
AMT_REQ_CREDIT_BUREAU_DAY       41519
AMT_REQ_CREDIT_BUREAU_WEEK      41519
AMT_REQ_CRED

- Several housing and property-related features contain extremely high missing values and may require removal or advanced imputation.
- EXT_SOURCE_1 and EXT_SOURCE_3 contain substantial missing data, reinforcing the importance of missing-value indicators and aggregated external score features.
- Employment-related engineered features inherited missing values from DAYS_EMPLOYED anomalies and require proper imputation handling.
- Bureau behavioral features contain limited missingness, likely due to applicants without historical bureau records.
- Social circle features contain relatively low missingness and may still provide useful behavioral risk information.
- A combination of feature removal, median imputation, mode imputation, and missing-value flags will be applied to handle missing data effectively.

## 8.2  DROP HIGH MISSING VALUE COLUMNS

In [ ]:
# DROP HIGH MISSING VALUE COLUMNS
high_missing_cols = ['COMMONAREA_AVG','NONLIVINGAPARTMENTS_AVG','LIVINGAPARTMENTS_AVG','FLOORSMIN_AVG','YEARS_BUILD_AVG','LANDAREA_AVG','BASEMENTAREA_AVG']
application_train.drop(columns=high_missing_cols,inplace=True)
print('High missing value columns dropped successfully!')

High missing value columns dropped successfully!


- Removed housing-related features with extremely high missing values to reduce noise and improve dataset quality.
- These columns contained insufficient information coverage and could negatively affect model stability and imputation reliability.
- Dropping high-missing features helps simplify the feature space and improve preprocessing efficiency.
- This step supports better model generalization by retaining more informative and reliable features.

In [ ]:
# ============================================================
# DROP HIGH-MISSING & LOW-IMPORTANCE COLUMNS
# ============================================================
drop_cols = ['APARTMENTS_AVG','YEARS_BEGINEXPLUATATION_AVG','ELEVATORS_AVG','ENTRANCES_AVG','FLOORSMAX_AVG','LIVINGAREA_AVG','NONLIVINGAREA_AVG','TOTALAREA_MODE',]
application_train.drop(columns=drop_cols,inplace=True)
print('High-missing columns dropped successfully!')
print('\n')
print(f'Updated Shape : {application_train.shape}')

High-missing columns dropped successfully!


Updated Shape : (307511, 99)


- Removed additional high-missing and low-importance housing-related features to improve dataset quality and reduce sparsity.
- These columns contained significant missing values and were less likely to contribute strong predictive power.
- Feature reduction helped simplify the dataset while retaining more reliable and informative variables.
- Removing highly sparse features can improve preprocessing efficiency and overall model robustness.

In [ ]:
# ============================================================
# DROP REMAINING OBJECT COLUMNS
# ============================================================
drop_object_cols = ['WEEKDAY_APPR_PROCESS_START','NAME_TYPE_SUITE','WALLSMATERIAL_MODE']
application_train.drop(columns=drop_object_cols,inplace=True)
print('Remaining object columns dropped successfully!')

Remaining object columns dropped successfully!


- Removed remaining low-priority object-type features to simplify preprocessing and encoding complexity.
- WEEKDAY_APPR_PROCESS_START and NAME_TYPE_SUITE contained limited additional predictive value after feature engineering.
- WALLSMATERIAL_MODE was removed due to high missing values and low reliability.
- Dropping unnecessary categorical features helps reduce dimensionality and improve overall model efficiency.

In [ ]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,ACTIVE_LOAN_RATIO,CLOSED_LOAN_RATIO,OVERDUE_PER_LOAN,PROLONGED_LOAN_RATIO,CREDIT_UTILIZATION_RATIO,AGE_YEARS,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO,TOTAL_SOCIAL_DEFAULTS,TOTAL_SOCIAL_OBS
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.375000,0.500000,0.0,0.0,0.410555,25.920548,1.745205,0.064828,4.0,4.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.222222,0.666667,0.0,0.0,0.284121,45.931507,3.254795,0.069352,0.0,2.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.200000,0.600000,0.0,0.0,0.000000,52.180822,0.616438,0.011591,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.000000,0.666667,0.0,0.0,0.000000,52.068493,8.326027,0.156892,0.0,4.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.500000,0.250000,0.0,0.0,0.864990,54.608219,8.323288,0.149677,0.0,0.0


## 8.3 OWN_CAR_AGE Imputation

In [ ]:
# OWN CAR AGE IMPUTATION
# Median car age for car owners
car_owner_median = application_train.loc[application_train['FLAG_OWN_CAR'] == 'Y','OWN_CAR_AGE'].median()
# Fill missing values only
# for car owners
application_train['OWN_CAR_AGE'] = np.where((application_train['FLAG_OWN_CAR'] == 'Y')&(application_train['OWN_CAR_AGE'].isnull()),car_owner_median,application_train['OWN_CAR_AGE'])
application_train[['FLAG_OWN_CAR','OWN_CAR_AGE']].head()

,FLAG_OWN_CAR,OWN_CAR_AGE
0,N,0.0
1,N,0.0
2,Y,26.0
3,N,0.0
4,N,0.0


- Applied conditional median imputation for OWN_CAR_AGE only on applicants who own a car.
- This approach preserves logical consistency by avoiding unrealistic imputation for non-car owners.
- Median imputation helps retain feature stability while reducing the impact of missing values.
- OWN_CAR_AGE may provide useful behavioral and financial insights related to asset ownership and applicant stability.

# 9 ) Encoding & Transformation

## 9.1 Binary Encoding

In [ ]:
# BINARY ENCODING
# Convert binary categorical features
# into numerical 0/1 format
application_train['CODE_GENDER'] = (application_train['CODE_GENDER'] == 'M').astype(int)
application_train['FLAG_OWN_CAR'] = (application_train['FLAG_OWN_CAR'] == 'Y').astype(int)
application_train['FLAG_OWN_REALTY'] = (application_train['FLAG_OWN_REALTY'] == 'Y').astype(int)
application_train['NAME_CONTRACT_TYPE'] = (application_train['NAME_CONTRACT_TYPE'] == 'Cash loans').astype(int)
# Display encoded features
application_train[['CODE_GENDER','FLAG_OWN_CAR','FLAG_OWN_REALTY','NAME_CONTRACT_TYPE']].head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_CONTRACT_TYPE
0,1,0,1,1
1,0,0,0,1
2,1,1,1,0
3,0,0,1,1
4,1,0,1,1


- Applied binary encoding to convert categorical yes/no and two-class features into numerical 0/1 representation.
- Binary encoding improves model compatibility and reduces preprocessing complexity for machine learning algorithms.
- Encoded features represent important applicant characteristics such as gender, property ownership, and loan contract type.
- Converting categorical variables into numerical format supports efficient training and better model interpretability.

## 9.2 One-Hot Encoding

In [ ]:
# CHECK UNIQUE CATEGORIES
# BEFORE ONE-HOT ENCODING
ohe_cols = ['NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE']
for col in ohe_cols:
    print('=' * 60)
    print(f'Column : {col}')
    print(f'\nTotal Unique Categories : {application_train[col].nunique()}')
    print('\nCategories :\n')
    print(application_train[col].value_counts())
    print('\n')

Column : NAME_INCOME_TYPE

Total Unique Categories : 8

Categories :

NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
Pensioner                55362
State servant            21703
Unemployed                  22
Student                     18
Businessman                 10
Maternity leave              5
Name: count, dtype: int64


Column : NAME_EDUCATION_TYPE

Total Unique Categories : 5

Categories :

NAME_EDUCATION_TYPE
Secondary / secondary special    218391
Higher education                  74863
Incomplete higher                 10277
Lower secondary                    3816
Academic degree                     164
Name: count, dtype: int64


Column : NAME_FAMILY_STATUS

Total Unique Categories : 6

Categories :

NAME_FAMILY_STATUS
Married                 196432
Single / not married     45444
Civil marriage           29775
Separated                19770
Widow                    16088
Unknown                      2
Name: count, dtype: int64


Column : NA

- Identified low-cardinality categorical features suitable for One-Hot Encoding.
- Most categories are concentrated in a few dominant groups, while some categories contain very few observations.
- Features such as income type, education level, family status, and housing type may provide important socio-economic and behavioral signals.
- One-Hot Encoding will convert these categorical variables into machine learning compatible numerical features while preserving category-level information.

In [ ]:
# ONE-HOT ENCODING
# Multi-category categorical columns
ohe_cols = ['NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE']
# Apply one-hot encoding
application_train = pd.get_dummies(application_train,columns=ohe_cols,drop_first=True,dtype=int)
print('One-Hot Encoding completed successfully!')
print(f'Updated Shape : {application_train.shape}')

One-Hot Encoding completed successfully!
Updated Shape : (307511, 113)


- Applied One-Hot Encoding to low-cardinality categorical features with multiple categories.
- Encoding transformed categorical variables into numerical indicator features suitable for machine learning models.
- drop_first=True was used to reduce multicollinearity and avoid the dummy variable trap.
- Dataset dimensionality increased from 99 to 113 columns after encoding, reflecting newly created category-level features.

In [ ]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Unknown,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents
0,100002,1,1,1,0,1,0,202500.0,406597.5,24700.5,...,0,0,1,0,0,1,0,0,0,0
1,100003,0,1,0,0,0,0,270000.0,1293502.5,35698.5,...,1,0,0,0,0,1,0,0,0,0
2,100004,0,0,1,1,1,0,67500.0,135000.0,6750.0,...,0,0,1,0,0,1,0,0,0,0
3,100006,0,1,0,0,1,0,135000.0,312682.5,29686.5,...,0,0,0,0,0,1,0,0,0,0
4,100007,0,1,1,0,1,0,121500.0,513000.0,21865.5,...,0,0,1,0,0,1,0,0,0,0


## 9.3 Frequency Encoding

In [ ]:
# CHECK UNIQUE CATEGORIES
# FOR FREQUENCY ENCODING
freq_cols = ['OCCUPATION_TYPE','ORGANIZATION_TYPE']
for col in freq_cols:
    print('=' * 60)
    print(f'Column : {col}')
    print(f'\nTotal Unique Categories : {application_train[col].nunique()}')
    print('\nTop Categories :\n')
    print(application_train[col].value_counts().head(20))
    print('\n')

Column : OCCUPATION_TYPE

Total Unique Categories : 18

Top Categories :

OCCUPATION_TYPE
Laborers                 55186
Sales staff              32102
Core staff               27570
Managers                 21371
Drivers                  18603
High skill tech staff    11380
Accountants               9813
Medicine staff            8537
Security staff            6721
Cooking staff             5946
Cleaning staff            4653
Private service staff     2652
Low-skill Laborers        2093
Waiters/barmen staff      1348
Secretaries               1305
Realty agents              751
HR staff                   563
IT staff                   526
Name: count, dtype: int64


Column : ORGANIZATION_TYPE

Total Unique Categories : 58

Top Categories :

ORGANIZATION_TYPE
Business Entity Type 3    67992
XNA                       55374
Self-employed             38412
Other                     16683
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School

- Identified high-cardinality categorical features suitable for Frequency Encoding.
- OCCUPATION_TYPE and ORGANIZATION_TYPE contain multiple categories with uneven category distributions.
- Frequency Encoding helps preserve category occurrence information while avoiding excessive dimensionality from One-Hot Encoding.
- This approach is especially useful for high-cardinality features and tree-based machine learning models.

In [ ]:
# FREQUENCY ENCODING
# High-cardinality categorical columns
freq_cols = ['OCCUPATION_TYPE','ORGANIZATION_TYPE']
# Apply frequency encoding
for col in freq_cols:
    freq_map = application_train[col].value_counts()
    application_train[col +'_FREQ'] = application_train[col].map(freq_map)
# Display encoded features
application_train[['OCCUPATION_TYPE_FREQ','ORGANIZATION_TYPE_FREQ']].head()

,OCCUPATION_TYPE_FREQ,ORGANIZATION_TYPE_FREQ
0,55186.0,67992
1,27570.0,8893
2,55186.0,10404
3,55186.0,67992
4,27570.0,85


- Applied Frequency Encoding to high-cardinality categorical features to avoid excessive dimensionality.
- Frequency Encoding replaces each category with its occurrence count in the dataset.
- OCCUPATION_TYPE and ORGANIZATION_TYPE may capture important employment and institutional risk patterns.
- This encoding approach is memory-efficient and particularly effective for tree-based machine learning algorithms.

In [ ]:
# ============================================================
# FINAL REMAINING MISSING VALUE HANDLING
# ============================================================
# Median fill columns
median_cols = ['CNT_FAM_MEMBERS','ANNUITY_INCOME_RATIO','GOODS_CREDIT_RATIO','INCOME_PER_PERSON','CHILDREN_RATIO']
for col in median_cols:
    application_train[col] = application_train[col].fillna(application_train[col].median())
# OCCUPATION_TYPE_FREQ → fill with 0
application_train['OCCUPATION_TYPE_FREQ'] = application_train['OCCUPATION_TYPE_FREQ'].fillna(0)
print('Final missing values handled successfully!')

Final missing values handled successfully!


- Applied median imputation to numerical engineered features with low missingness to preserve distribution stability.
- OCCUPATION_TYPE_FREQ missing values were filled with 0 to represent unseen or unavailable occupation categories.
- Final missing value handling ensures improved dataset completeness before model training.
- Proper imputation helps maintain model stability, reduce data loss, and support efficient machine learning preprocessing.

In [ ]:
# ============================================================
# DROP REDUNDANT OBJECT COLUMNS
# ============================================================
application_train.drop(
    columns=['OCCUPATION_TYPE','ORGANIZATION_TYPE'],inplace=True)
print('Redundant object columns dropped successfully!')

Redundant object columns dropped successfully!


- Dropped original high-cardinality object columns after applying Frequency Encoding.
- Removing redundant categorical columns helps reduce memory usage and preprocessing complexity.
- Encoded frequency-based features retain important category distribution information in numerical format.
- This step improves dataset cleanliness and prepares the data for efficient machine learning model training.

# 10 ) Data Validation & Final Cleaning

## 10.1 DAYS_BIRTH Sanity Check

In [ ]:
# DAYS_BIRTH SANITY CHECK
print(application_train['DAYS_BIRTH'].describe())
print('\n')
print('All values negative :',(application_train['DAYS_BIRTH'] < 0).all())

count    307511.000000
mean     -16036.995067
std        4363.988632
min      -25229.000000
25%      -19682.000000
50%      -15750.000000
75%      -12413.000000
max       -7489.000000
Name: DAYS_BIRTH, dtype: float64


All values negative : True


- DAYS_BIRTH contains only negative values, which correctly represent days before the current application date.
- The feature shows realistic age distributions with no obvious anomalies or invalid positive values.
- Mean applicant age is approximately 44 years, based on day-to-year conversion.
- DAYS_BIRTH is suitable for further feature engineering such as AGE_YEARS and age-based behavioral ratios.

## 10.2 INCOME CAPPING VERIFICATION

In [ ]:
# INCOME CAPPING VERIFICATION
print('Maximum Income :',application_train['AMT_INCOME_TOTAL'].max())

Maximum Income : 472500.0


- Income capping was successfully applied to control extreme income outliers.
- The maximum annual income value was reduced to 472500, improving distribution stability.
- Outlier treatment helps reduce skewness and prevents extreme values from disproportionately influencing the model.
- Controlled income distributions can improve model robustness and training performance.

## 10.3 MEDIAN IMPUTATION

In [ ]:
median_fill_cols = ['AMT_ANNUITY','AMT_GOODS_PRICE','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','OBS_30_CNT_SOCIAL_CIRCLE','DEF_30_CNT_SOCIAL_CIRCLE','OBS_60_CNT_SOCIAL_CIRCLE','DEF_60_CNT_SOCIAL_CIRCLE','DAYS_LAST_PHONE_CHANGE','AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_DAY','AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR','EXT_SOURCE_MEAN','ACTIVE_LOAN_RATIO','CLOSED_LOAN_RATIO','OVERDUE_PER_LOAN','PROLONGED_LOAN_RATIO','CREDIT_UTILIZATION_RATIO']
for col in median_fill_cols:
    application_train[col] = application_train[col].fillna(application_train[col].median())

- Applied median imputation to numerical features with missing values to improve dataset completeness and stability.
- Median imputation was preferred because it is robust to skewed distributions and extreme outliers.
- Missing values were handled across credit bureau, social risk, external score, and engineered behavioral features.
- Proper missing value treatment helps prevent information loss and supports stable machine learning model performance.

## 10.4 ZERO IMPUTATION

In [ ]:
# ============================================================
# ZERO IMPUTATION
# ============================================================
zero_fill_cols = ['DAYS_EMPLOYED','EMPLOYMENT_YEARS','EMPLOYMENT_AGE_RATIO','TOTAL_SOCIAL_DEFAULTS','TOTAL_SOCIAL_OBS','EXT_SOURCE_STD']
for col in zero_fill_cols:
    application_train[col] = application_train[col].fillna(0)
print('Zero imputation completed successfully!')

Zero imputation completed successfully!


- Applied zero imputation to features where missing values logically represent absence of activity or unavailable behavioral information.
- Employment-related features were filled with 0 to handle missing employment duration after anomaly treatment.
- Social risk and bureau behavioral features were also zero-imputed to represent no observed defaults, social observations, or bureau activity.
- Zero imputation helps preserve dataset consistency while maintaining meaningful behavioral interpretations.

##10.5 FINAL MISSING VALUE CHECK

In [ ]:
# ============================================================
# FINAL MISSING VALUE CHECK
# ============================================================
missing = application_train.isnull().sum()
missing = missing[missing > 0]
print(missing)

Series([], dtype: int64)


- Final missing value validation was performed to ensure dataset completeness after all preprocessing and imputation steps.
- All remaining missing values were successfully handled using appropriate median and zero imputation strategies.
- The dataset is now fully cleaned, feature engineered, and ready for machine learning model training.
- Final validation confirms improved data quality and preprocessing consistency across all features.

## 10.6  INFINITE VALUE CHECK

In [ ]:
# ============================================================
# INFINITE VALUE CHECK
# ============================================================
import numpy as np
inf_values = np.isinf(application_train.select_dtypes(include=['float64', 'int64'])).sum().sum()
print(f'Total Infinite Values : {inf_values}')

Total Infinite Values : 0


- Performed infinite value validation to identify invalid numerical values generated during feature engineering.
- The check confirms whether any division-based engineered features produced positive or negative infinity values.
- Infinite value validation is important for ensuring model stability and preventing training errors.
- Proper denominator safeguards (+1 protection) helped maintain numerical consistency across engineered ratio features.

In [ ]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Unknown,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_FREQ,ORGANIZATION_TYPE_FREQ
0,100002,1,1,1,0,1,0,202500.0,406597.5,24700.5,...,1,0,0,1,0,0,0,0,55186.0,67992
1,100003,0,1,0,0,0,0,270000.0,1293502.5,35698.5,...,0,0,0,1,0,0,0,0,27570.0,8893
2,100004,0,0,1,1,1,0,67500.0,135000.0,6750.0,...,1,0,0,1,0,0,0,0,55186.0,10404
3,100006,0,1,0,0,1,0,135000.0,312682.5,29686.5,...,0,0,0,1,0,0,0,0,55186.0,67992
4,100007,0,1,1,0,1,0,121500.0,513000.0,21865.5,...,1,0,0,1,0,0,0,0,27570.0,85


## 10.7 CONSTANT COLUMN CHECK

In [ ]:
# ============================================================
# CONSTANT COLUMN CHECK
# ============================================================
constant_cols = []
for col in application_train.columns:
    if application_train[col].nunique() == 1:
        constant_cols.append(col)
print(constant_cols)

[]


- Performed constant column validation to identify features containing only a single unique value.
- Constant features provide no predictive information and may unnecessarily increase dataset dimensionality.
- Removing constant columns helps improve model efficiency and reduces redundant feature space.
- This validation step ensures higher data quality and cleaner machine learning preprocessing.

## 10.8  HIGH CORRELATION CHECK

In [ ]:
# ============================================================
# HIGH CORRELATION CHECK
# ============================================================
corr_matrix = application_train.corr(numeric_only=True).abs()
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if corr_matrix.iloc[i, j] > 0.90:
            high_corr.append((corr_matrix.columns[i],corr_matrix.columns[j],corr_matrix.iloc[i, j]))
print(high_corr[:20])

[('AMT_GOODS_PRICE', 'AMT_CREDIT', np.float64(0.9861853684384371)), ('REGION_RATING_CLIENT_W_CITY', 'REGION_RATING_CLIENT', np.float64(0.9508422141646482)), ('OBS_60_CNT_SOCIAL_CIRCLE', 'OBS_30_CNT_SOCIAL_CIRCLE', np.float64(0.9984912890365444)), ('DAYS_EMPLOYED_ANOMALY', 'FLAG_EMP_PHONE', np.float64(0.9998678692146539)), ('CHILDREN_RATIO', 'CNT_CHILDREN', np.float64(0.9710181117826725)), ('AVG_CREDIT_PER_LOAN', 'AMT_CREDIT_SUM_mean', np.float64(0.9748120387529589)), ('DEBT_PER_LOAN', 'AMT_CREDIT_SUM_DEBT_mean', np.float64(0.9521201204539547)), ('AGE_YEARS', 'DAYS_BIRTH', np.float64(0.9999999999999887)), ('EMPLOYMENT_YEARS', 'DAYS_EMPLOYED', np.float64(0.9999999999999691)), ('EMPLOYMENT_AGE_RATIO', 'DAYS_EMPLOYED', np.float64(0.9633043786737179)), ('EMPLOYMENT_AGE_RATIO', 'EMPLOYMENT_YEARS', np.float64(0.9633043786737373)), ('TOTAL_SOCIAL_DEFAULTS', 'DEF_30_CNT_SOCIAL_CIRCLE', np.float64(0.9716565040670403)), ('TOTAL_SOCIAL_DEFAULTS', 'DEF_60_CNT_SOCIAL_CIRCLE', np.float64(0.9565748732

- Performed correlation analysis to identify highly correlated numerical features in the dataset.
- Features with correlation greater than 0.90 may contain redundant information and increase multicollinearity.
- Highly correlated features can negatively impact model interpretability and may lead to overfitting in certain algorithms.
- Correlation analysis helps support feature selection and improve overall model efficiency.

## 10.9 DROP HIGHLY CORRELATED / REDUNDANT FEATURES

In [ ]:
# ============================================================
# DROP HIGHLY CORRELATED / REDUNDANT FEATURES
# ============================================================
drop_high_corr = ['DAYS_BIRTH','DAYS_EMPLOYED','FLAG_EMP_PHONE','NAME_INCOME_TYPE_Pensioner']
# Drop columns safely
application_train.drop(columns=[
col for col in drop_high_corr
if col in application_train.columns
],inplace=True)
# ============================================================
# DISPLAY UPDATED SHAPE
# ============================================================
print(f'Dropped Columns : {len(drop_high_corr)}')
print(f'Updated Shape : {application_train.shape}')

Dropped Columns : 4
Updated Shape : (307511, 109)


- Removed highly correlated and redundant features identified through correlation analysis.
- Exact transformed duplicates and near-identical behavioral indicators were dropped to reduce redundancy.
- Important engineered financial and behavioral features were retained to preserve predictive information.
- Feature reduction helps improve model efficiency, interpretability, and preprocessing quality.

## 10.10 FINAL DATASET INFO

In [ ]:
# ============================================================
# FINAL DATASET INFO
# ============================================================
application_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 112 entries, SK_ID_CURR to RECENT_INQUIRY_RATIO
dtypes: float64(52), int64(60)
memory usage: 262.8 MB


- Final dataset validation confirms successful completion of data cleaning, feature engineering, encoding, and preprocessing steps.
- The dataset now contains fully transformed numerical and machine learning compatible features.
- Redundant, highly correlated, and high-missing-value features were removed to improve dataset quality and efficiency.
- Engineered behavioral, financial, bureau, social-risk, and external credit score features were successfully integrated into the final dataset.
- The dataset is now ML-ready and prepared for train-test splitting, model training, and evaluation.

## 10.11 Final Feature Engineering Summary

In [ ]:
# ============================================================
# FINAL FEATURE ENGINEERING SUMMARY
# ============================================================
 # Financial Features
new_features = ['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','GOODS_CREDIT_RATIO','CREDIT_ANNUITY_RATIO',
    # Family Features
    'INCOME_PER_PERSON','CHILDREN_RATIO',
    # Document Features
    'TOTAL_DOCUMENTS_SUBMITTED',
    # Bureau Features
    'BUREAU_DEBT_RATIO','AVG_CREDIT_PER_LOAN','DEBT_PER_LOAN',
    # Bureau Behavioral Features
    'ACTIVE_LOAN_RATIO','CLOSED_LOAN_RATIO','OVERDUE_PER_LOAN','PROLONGED_LOAN_RATIO','CREDIT_UTILIZATION_RATIO',
    # Inquiry Features
    'TOTAL_INQUIRIES','RECENT_INQUIRY_RATIO',
    # Behavioral Features
    'AGE_YEARS','EMPLOYMENT_YEARS','EMPLOYMENT_AGE_RATIO',
    # Social Risk Features
    'TOTAL_SOCIAL_DEFAULTS','TOTAL_SOCIAL_OBS',
    # External Credit Features
    'EXT_SOURCE_MEAN','EXT_SOURCE_STD','EXT_SOURCE_1_MISSING',
    # Collateral Feature
    'HAS_COLLATERAL'
]
print('=' * 65)
print('FINAL FEATURE ENGINEERING SUMMARY')
print('=' * 65)
print('\n')
print(f'Total Engineered Features : {len(new_features)}')
print(f'Final Dataset Shape : {application_train.shape}')
print('\n')
print('Engineered Features List:\n')
for i, feature in enumerate(
    new_features,
    1
):
    print(f'{i}. {feature}')
print('\n')
print('=' * 65)
print('Successfully engineered advanced predictive features')
print('for AI-driven loan recovery probability modeling.')
print('Dataset is fully cleaned, transformed, validated,')
print('and ready for Machine Learning model training.')
print('=' * 65)

FINAL FEATURE ENGINEERING SUMMARY


Total Engineered Features : 26
Final Dataset Shape : (307511, 112)


Engineered Features List:

1. CREDIT_INCOME_RATIO
2. ANNUITY_INCOME_RATIO
3. GOODS_CREDIT_RATIO
4. CREDIT_ANNUITY_RATIO
5. INCOME_PER_PERSON
6. CHILDREN_RATIO
7. TOTAL_DOCUMENTS_SUBMITTED
8. BUREAU_DEBT_RATIO
9. AVG_CREDIT_PER_LOAN
10. DEBT_PER_LOAN
11. ACTIVE_LOAN_RATIO
12. CLOSED_LOAN_RATIO
13. OVERDUE_PER_LOAN
14. PROLONGED_LOAN_RATIO
15. CREDIT_UTILIZATION_RATIO
16. TOTAL_INQUIRIES
17. RECENT_INQUIRY_RATIO
18. AGE_YEARS
19. EMPLOYMENT_YEARS
20. EMPLOYMENT_AGE_RATIO
21. TOTAL_SOCIAL_DEFAULTS
22. TOTAL_SOCIAL_OBS
23. EXT_SOURCE_MEAN
24. EXT_SOURCE_STD
25. EXT_SOURCE_1_MISSING
26. HAS_COLLATERAL


Successfully engineered advanced predictive features
for AI-driven loan recovery probability modeling.
Dataset is fully cleaned, transformed, validated,
and ready for Machine Learning model training.


# 11 )  SAVE &  DOWNLOAD FINAL DATASET

In [ ]:
# ============================================================
# SAVE FINAL DATASET
# ============================================================
application_train.to_csv('final_feature_engineered_dataset.csv',index=False)
print('Dataset saved successfully!')

Dataset saved successfully!


- Saved the fully cleaned and feature-engineered dataset as final_feature_engineered_dataset.csv.
- The final dataset contains transformed, encoded, validated, and ML-ready features for downstream modeling tasks.
- Exporting the processed dataset enables reproducible model training, evaluation, and deployment workflows.
- The dataset is now prepared for train-test splitting, machine learning model development, and explainability analysis.